# 02 — Invent 40 patients

`oracle()` gives the right answers. `render()` writes the text.

**Write `oracle()` from the policy rules and then don't touch it.** Adjusting the answer key
after seeing what the model said is the easiest way to accidentally cheat, and it's very
tempting around week three.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
import random

BUCKETS = {
    "clear_met":      8,   # AHI 18-40, 30+ events, everything documented
    "met_5_14":       6,   # AHI 7-13, 10+ events, one listed symptom
    "clear_unmet":    6,   # AHI 3, no symptoms; E0471 with OSA as primary dx
    "insufficient":   8,   # AHI missing, events missing, adherence never recorded
    "borderline":     8,   # AHI 14 + symptom; AHI 15 but 25 events; 68% of nights; day 95
    "adversarial":    4,   # denial cites wrong rule; record has a near-miss quote as bait
}
assert sum(BUCKETS.values()) == 40

SYMPTOMS = ["excessive daytime sleepiness", "impaired cognition", "mood disorder",
            "insomnia", "hypertension", "ischemic heart disease", "history of stroke"]


## The answer key

Straight from the rules. `None` means "the record doesn't say" — which is
`insufficient_evidence`, never `unmet`. Getting that distinction right here is the whole
project.

In [ ]:
def oracle(spec):
    """spec -> {criterion_id: label}. Plain if-statements, no cleverness."""
    out = {}

    # B1: AHI >= 15 with >= 30 events
    if spec["ahi"] is None or spec["events"] is None:
        out["B1"] = "insufficient_evidence"
    elif spec["ahi"] >= 15 and spec["events"] >= 30:
        out["B1"] = "met"
    else:
        out["B1"] = "unmet"

    # TODO B2, B2_symptoms, adherence, reeval_window, E0470_trial, E0471_osa
    # Same shape every time: missing data -> insufficient_evidence, then the numeric test.

    return out


## The text\n\n3-4 phrasings per field so all 40 don't read identically.

In [ ]:
AHI_PHRASES = [
    "AHI {ahi} events/hour, {events} total respiratory events recorded.",
    "Apnea-hypopnea index calculated at {ahi}/hr over {events} scored events.",
    "Study shows an AHI of {ahi} per hour ({events} events).",
]

def render(spec, rng):
    """spec -> {'sleep_study', 'chart_note', 'denial_letter'}"""
    raise NotImplementedError


## Build all 40

In [ ]:
def make_spec(bucket, i, rng):
    """bucket name -> a spec dict. One branch per bucket."""
    raise NotImplementedError

rng = random.Random(0)
cases = []
i = 0
for bucket, n in BUCKETS.items():
    for _ in range(n):
        i += 1
        spec = make_spec(bucket, i, rng)
        cases.append({"id": f"case_{i:03d}", "bucket": bucket, "spec": spec,
                      "documents": render(spec, rng), "gold": oracle(spec),
                      "hand_written": False})

json.dump(cases, open("data/cases.json", "w"), indent=2)
len(cases)


## Then write 10 by hand

Open `data/cases.json`, pick 10, and rewrite the text in my own words with no template. Set
`hand_written: true` on those. Score them separately in notebook 05 — if the model does much
worse on them, my templates were too easy and the README has to say so.

## Sanity-check my answer key

Get a classmate to label 10 cases blind, then compare. This checks whether *I* understood the
policy, which is the right thing to be checking.

In [ ]:
from sklearn.metrics import cohen_kappa_score

classmate = []   # their labels, in case order
mine      = []   # oracle() labels for the same cases
# cohen_kappa_score(classmate, mine)


## Read a few out loud

In [ ]:
cases = json.load(open("data/cases.json"))
from collections import Counter
print(Counter(c["bucket"] for c in cases))
print(cases[0]["documents"]["sleep_study"])
print(cases[0]["gold"])
